# Snake Game Using Pygame

This project implements a Snake game using the Pygame library. The player controls a snake that moves around the game window, collects apples, and increases its score and length.

The game also includes a second snake that becomes active after the player's score exceeds 10. The second snake attempts to follow the apple, creating an additional challenge.

Additional objects, including a bonus item and an obstacle, are also included.

## 1. Install and Import Libraries

Install Pygame and import the required Python libraries. Pygame is used for the game window, graphics, keyboard input, and sound.

In [ ]:
!pip install pygame

import os
import random
import pygame

## 2. Game Initialization

Initialize Pygame and define the game window, dimensions, movement speed, colors, and asset directory. The game uses a grid-based movement system.

In [ ]:
pygame.init()

try:
    pygame.mixer.init()
    SOUND_ENABLED = True
except pygame.error:
    SOUND_ENABLED = False

SCREEN_WIDTH = 1000
SCREEN_HEIGHT = 1000
BLOCK_SIZE = 100
FPS = 10

screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
pygame.display.set_caption("Snake")

clock = pygame.time.Clock()
font = pygame.font.SysFont(None, 25)

red = (220, 20, 60)
white = (255, 255, 255)
background = (200, 200, 200)

ASSET_DIR = os.path.join(os.getcwd(), "assets")

if not os.path.isdir(ASSET_DIR):
    raise FileNotFoundError("The assets directory was not found. Create an assets folder next to the notebook.")

## 3. Load Game Assets

Load the images used for the snakes, apple, bonus item, and obstacle. The eating sound is loaded separately and is optional if audio is unavailable.

In [ ]:
def load_image(filename, size=(100, 100)):
    path = os.path.join(ASSET_DIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing asset: {path}")
    image = pygame.image.load(path).convert_alpha()
    return pygame.transform.scale(image, size)


image = load_image("circle.png")
image2 = load_image("Golden_Big_Chain_Chomp_Sprite_PM5.png")
image3 = load_image("Featured-Image.png")
image4 = load_image("super_mario_world_mushroom_sprite_by_adventuresmg64gaming-db12tjs.png")
apple_image = load_image("apple90.png")

icon_path = os.path.join(ASSET_DIR, "Golden_Big_Chain_Chomp_Sprite_PM5.png")
pygame.display.set_icon(pygame.image.load(icon_path).convert_alpha())

EAT_SOUND_PATH = os.path.join(ASSET_DIR, "burger (1).wav")
eat_sound = None

if SOUND_ENABLED and os.path.exists(EAT_SOUND_PATH):
    eat_sound = pygame.mixer.Sound(EAT_SOUND_PATH)

print("Game assets loaded successfully.")

## 4. Helper Functions

Define reusable functions for generating random positions, creating snake segments, displaying messages and scores, drawing snakes, and spawning apples. All object positions are aligned with the movement grid.

In [ ]:
def random_position():
    x = random.randrange(100, 900, BLOCK_SIZE)
    y = random.randrange(100, 900, BLOCK_SIZE)
    return x, y


def create_segment(x, y):
    rect = image.get_rect()
    rect.x = x
    rect.y = y
    return rect


def message(msg, color, y_displace=0):
    text_surface = font.render(msg, True, color)
    text_rect = text_surface.get_rect()
    text_rect.center = (SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2 + y_displace)
    screen.blit(text_surface, text_rect)


def draw_score(value, x=0):
    text = font.render("Score: " + str(value), True, red)
    screen.blit(text, (x, 0))


def draw_snake(body, head_image, body_image):
    for segment in body[:-1]:
        screen.blit(body_image, segment)
    if body:
        screen.blit(head_image, body[-1])


def spawn_apple(apple):
    apple.x, apple.y = random_position()


def play_eat_sound():
    if eat_sound is not None:
        eat_sound.play()

## 5. Initialize Game Objects

Create the player snake and the second snake using lists of rectangular body segments. Initialize the apple, bonus item, obstacle, scores, directions, and game-state variables.

In [ ]:
player_body = [
    create_segment(700, 400),
    create_segment(600, 400),
    create_segment(500, 400),
    create_segment(400, 400)
]

second_body = [
    create_segment(800, 500),
    create_segment(700, 500),
    create_segment(600, 500),
    create_segment(500, 500)
]

apple = apple_image.get_rect()
spawn_apple(apple)

bonus = image4.get_rect()
bonus.x, bonus.y = random_position()

obstacle = image3.get_rect()
obstacle.x, obstacle.y = 200, 200

player_score = 0
second_score = 0

# Directions: 0 = down, 1 = right, 2 = up, 3 = left
player_direction = 1
second_direction = 1

game_running = True
game_started = False

## 6. Welcome Screen

Display the welcome screen before starting the game. Press P to start, Q to quit, or Escape to exit.

In [ ]:
screen.fill(white)
message("Welcome to Snake", red, -90)
message("Eat the Apple!", red, -40)
message("Press P to play or Q to quit!", red, -10)
pygame.display.update()

while not game_started and game_running:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            game_running = False
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_p:
                game_started = True
            elif event.key in (pygame.K_q, pygame.K_ESCAPE):
                game_running = False

    clock.tick(30)

## 7. Player Controls and Movement

The player controls the snake using the arrow keys. The snake moves continuously in its current direction. Immediate reversal is prevented to avoid moving directly into its own body.

The movement function adds a new head segment and removes the tail unless the snake is growing.

In [ ]:
def handle_player_direction():
    global player_direction

    keys = pygame.key.get_pressed()

    if keys[pygame.K_LEFT] and player_direction in (0, 2):
        player_direction = 3
    elif keys[pygame.K_RIGHT] and player_direction in (0, 2):
        player_direction = 1
    elif keys[pygame.K_DOWN] and player_direction in (1, 3):
        player_direction = 0
    elif keys[pygame.K_UP] and player_direction in (1, 3):
        player_direction = 2


def get_movement(direction):
    if direction == 0:
        return 0, BLOCK_SIZE
    elif direction == 1:
        return BLOCK_SIZE, 0
    elif direction == 2:
        return 0, -BLOCK_SIZE
    elif direction == 3:
        return -BLOCK_SIZE, 0
    return 0, 0


def move_snake(body, direction, grow=False):
    move_x, move_y = get_movement(direction)
    new_head = create_segment(body[-1].x + move_x, body[-1].y + move_y)
    body.append(new_head)

    if not grow:
        del body[0]

    return new_head

## 8. Second Snake Movement

The second snake becomes active after the player's score exceeds 10. It changes its direction to move toward the apple while avoiding an immediate reversal.

In [ ]:
def update_second_direction():
    global second_direction

    head = second_body[-1]
    dx = apple.x - head.x
    dy = apple.y - head.y

    if second_direction in (1, 3):
        if dy > 0:
            second_direction = 0
        elif dy < 0:
            second_direction = 2
        elif dx > 0:
            second_direction = 1
        elif dx < 0:
            second_direction = 3
    else:
        if dx > 0:
            second_direction = 1
        elif dx < 0:
            second_direction = 3
        elif dy > 0:
            second_direction = 0
        elif dy < 0:
            second_direction = 2

## 9. Main Game Loop

Run the game continuously while it is active. Each iteration processes user input, updates the snakes, checks collisions with the apple and bonus item, updates scores, and redraws the game window.

When a snake eats the apple, it grows by one segment. The second snake is activated when the player's score exceeds 10.

In [ ]:
while game_running and game_started:
    clock.tick(FPS)

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            game_running = False
        elif event.type == pygame.KEYDOWN:
            if event.key in (pygame.K_ESCAPE, pygame.K_q):
                game_running = False

    if not game_running:
        break

    screen.fill(background)

    # Draw static objects.
    screen.blit(apple_image, apple)
    screen.blit(image3, obstacle)
    screen.blit(image4, bonus)

    # Update player snake.
    handle_player_direction()
    move_x, move_y = get_movement(player_direction)
    next_player_head = create_segment(
        player_body[-1].x + move_x,
        player_body[-1].y + move_y
    )
    player_eats_apple = next_player_head.colliderect(apple)
    player_head = move_snake(player_body, player_direction, grow=player_eats_apple)

    if player_eats_apple:
        player_score += 10
        play_eat_sound()
        spawn_apple(apple)

    if player_head.colliderect(bonus):
        player_score -= 10
        bonus.x, bonus.y = random_position()

    # Update second snake after the score threshold is reached.
    if player_score > 10:
        update_second_direction()

        move_x, move_y = get_movement(second_direction)
        next_second_head = create_segment(
            second_body[-1].x + move_x,
            second_body[-1].y + move_y
        )
        second_eats_apple = next_second_head.colliderect(apple)
        second_head = move_snake(
            second_body, second_direction, grow=second_eats_apple
        )

        if second_eats_apple:
            second_score += 10
            play_eat_sound()
            spawn_apple(apple)

        if second_head.colliderect(bonus):
            bonus.x, bonus.y = random_position()

    # Collision detection and game-over conditions.
    if player_head.colliderect(obstacle) or player_score <= -50:
        screen.fill(white)
        message("Game Over!", red, -90)
        pygame.display.update()
        pygame.time.wait(1500)
        game_running = False
        break

    # Draw snakes and scores.
    draw_snake(player_body, image2, image)

    if player_score > 10:
        draw_snake(second_body, image2, image)

    draw_score(player_score, 0)

    if player_score > 10:
        draw_score(second_score, 100)

    pygame.display.update()

## 10. Quit Pygame

Close the Pygame window and release the resources after the game loop ends.

In [ ]:
pygame.quit()

## Conclusion

The project demonstrates the implementation of a graphical Snake game using Pygame. It includes keyboard-based player control, grid movement, apple collection, score tracking, a second snake that follows the apple, bonus items, obstacle collision, and game-over handling.